# Notebook 02 — Traitement avec PySpark
## Projet fil rouge : BigData Commerce Analytics · Jour 2 · B2 Sup de Vinci

**Objectif :** lire les collections MongoDB `raw_commandes` et `raw_payments` avec Spark, réaliser des transformations et des actions, comprendre la lazy evaluation, observer Spark UI, puis calculer les 4 indicateurs du projet.

**Prérequis :** Java 17, PySpark 4.0.1, MongoDB démarré (`docker start mongo`) et le notebook 01 exécuté.

> Au premier lancement, Spark télécharge le connecteur MongoDB depuis Internet : la cellule de démarrage peut prendre 1 à 2 minutes.

## Atelier 2 (1/2) — Démarrer Spark en mode local

- `local[*]` : Spark tourne sur ce Mac et utilise tous les cœurs du processeur.
- `spark.jars.packages` : charge le **connecteur MongoDB 11.x**, compatible avec Spark 4 (Scala 2.13).

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MONGO_URI = "mongodb://localhost:27017"

spark = (SparkSession.builder
    .appName("BigDataCommerceAnalytics")
    .master("local[*]")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .getOrCreate())

print("Version PySpark :", pyspark.__version__)
print("Spark UI :", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/27 11:30:25 WARN Utils: Your hostname, Abdou, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/27 11:30:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/mouad/Downloads/BigData_Commerce_Analytics_DummyJSON/BigData_Commerce_Analytics_DummyJSON/venv_linux/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/admin/.ivy2.5.2/cache
The jars for the packages stored in: /home/admin/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1069ad79-51b9-45cb-a985-eaecad4842bc;1.0
	confs: [default]


	found org.mongodb.spark#mongo-spark-connector_2.13;11.1.0 in central


	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central


	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central


... (4 outputs left out)

	[SUCCESSFUL ] org.mongodb#mongodb-driver-core;5.1.4!mongodb-driver-core.jar (478ms)
downloading https://repo1.maven.org/maven2/org/mongodb/bson-record-codec/5.1.4/bson-record-codec-5.1.4.jar ...
	[SUCCESSFUL ] org.mongodb#bson-record-codec;5.1.4!bson-record-codec.jar (92ms)
:: resolution report :: resolve 4364ms :: artifacts dl 1437ms
	:: modules in use:
	org.mongodb#bson;5.1.4 from central in [default]
	org.mongodb#bson-record-codec;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-core;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-sync;5.1.4 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.13;11.1.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   5   |   5   

26/11/27 11:30:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Version PySpark : 4.0.1
Spark UI : http://10.255.255.254:4040


## Atelier 2 (2/2) — 1. Lire les données depuis MongoDB

In [2]:
def lire_collection(nom):
    return (spark.read.format("mongodb")
            .option("database", "commerce")
            .option("collection", nom)
            .load())

commandes = lire_collection("raw_commandes")
payments = lire_collection("raw_payments")

commandes.printSchema()
payments.printSchema()

root
 |-- _id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cart_discounted_total: double (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- cart_total: double (nullable = true)
 |-- category: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discounted_line_total: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)

root
 |-- _id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment_date: string (nullable = true)
 |-- payment_id: string (nullable = true)
 |-- simulated: boolean 

In [3]:
commandes.show(5)
payments.show(5)
print("Documents lus dans raw_commandes :", commandes.count())
print("Documents lus dans raw_payments  :", payments.count())

+--------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+----------+----------+----------+--------------------+--------+----------+-------+--------+
|                 _id|         brand|cart_discounted_total|cart_id|cart_total|          category|discount_percentage|discounted_line_total|       line_total|order_date|  order_id|product_id|       product_title|quantity|unit_price|user_id|username|
+--------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+----------+----------+----------+--------------------+--------+----------+-------+--------+
|6b095b79440970167...|          NULL|             11510.81|      1|  13037.88|              tops|              12.13|               105.41|           119.96|2025-03-02|CART-00001|       162|          Blue Frock|       4|     29.99|      1|  emilys|
|6b0

Documents lus dans raw_commandes : 800
Documents lus dans raw_payments  : 208


**Le champ `_id` :** c'est l'identifiant technique que MongoDB ajoute automatiquement à chaque document (type ObjectId). Spark le lit comme une colonne ordinaire. Il ne sert pas à l'analyse métier : on utilise nos propres identifiants (`cart_id`, `order_id`, `user_id`), donc on peut l'écarter avec `select` ou `drop`.

## 2. Transformations et actions

On enchaîne des **transformations** (select, filter, withColumn, groupBy, orderBy) puis on déclenche le calcul avec des **actions** (show, count, collect).

In [4]:
# Transformations : sélectionner, filtrer, créer une colonne
lignes = (commandes
    .select("cart_id", "user_id", "category", "quantity", "line_total", "discounted_line_total")
    .filter(F.col("quantity") >= 2)
    .withColumn("montant_remise", F.round(F.col("line_total") - F.col("discounted_line_total"), 2)))

# Transformations : regrouper, agréger, trier
par_categorie = (lignes
    .groupBy("category")
    .agg(F.count("*").alias("nb_lignes"),
         F.sum("quantity").alias("quantite_totale"),
         F.round(F.sum("montant_remise"), 2).alias("remises_accordees"))
    .orderBy(F.desc("remises_accordees")))

# Actions : c'est ici que Spark calcule
par_categorie.show(10)
print("Lignes avec au moins 2 articles :", lignes.count())
print("Première catégorie :", par_categorie.collect()[0]["category"])

+------------------+---------+---------------+-----------------+
|          category|nb_lignes|quantite_totale|remises_accordees|
+------------------+---------+---------------+-----------------+
|           vehicle|       16|             57|        185544.15|
|        motorcycle|       21|             82|         58790.75|
|      mens-watches|       20|             66|         45805.44|
|    womens-watches|       10|             37|          28890.5|
|         furniture|       19|             70|         10296.98|
|       smartphones|       47|            164|          8561.34|
|           laptops|       14|             54|          6561.36|
|mobile-accessories|       57|            185|          3261.69|
|           tablets|       12|             38|          1815.88|
|       womens-bags|       18|             59|           803.82|
+------------------+---------+---------------+-----------------+
only showing top 10 rows


Lignes avec au moins 2 articles : 648


Première catégorie : vehicle


| Transformations utilisées (rien n'est calculé) | Actions utilisées (le calcul démarre) |
|---|---|
| `select`, `filter`, `withColumn`, `groupBy` + `agg`, `orderBy` | `show`, `count`, `collect` |

**Différence :** une transformation décrit une étape et renvoie un nouveau DataFrame, sans lire les données. Une action demande un résultat (afficher, compter, ramener des lignes) : elle oblige Spark à exécuter toutes les transformations qui la précèdent.

## 3. Comprendre la lazy evaluation

1. Exécutez la cellule suivante : elle définit 3 transformations, **sans action**.
2. Ouvrez Spark UI (adresse affichée au démarrage, port 4040), onglet **Jobs** : aucun nouveau job n'apparaît.
3. Exécutez la cellule d'après (action `count`), puis rafraîchissez Spark UI : un nouveau job est apparu.

In [5]:
etape1 = commandes.filter(F.col("category") == "laptops")
etape2 = etape1.withColumn("prix_ttc", F.round(F.col("unit_price") * 1.2, 2))
etape3 = etape2.select("product_title", "unit_price", "prix_ttc")
print("3 transformations définies : aucun calcul n'a encore eu lieu.")

# Spark a seulement préparé un plan d'exécution :
etape3.explain()

3 transformations définies : aucun calcul n'a encore eu lieu.
== Physical Plan ==
*(1) Project [product_title#12, unit_price#14, round((unit_price#14 * 1.2), 2) AS prix_ttc#477]
+- BatchScan MongoTable()[product_title#12, unit_price#14] MongoScan{namespaceDescription=commerce.raw_commandes} RuntimeFilters: []




In [6]:
# Action : Spark exécute maintenant le plan complet
print("Lignes 'laptops' :", etape3.count())
etape3.show(5)

Lignes 'laptops' : 16


+--------------------+----------+--------+
|       product_title|unit_price|prix_ttc|
+--------------------+----------+--------+
|Asus Zenbook Pro ...|   1799.99| 2159.99|
|     Lenovo Yoga 920|   1099.99| 1319.99|
|Apple MacBook Pro...|   1999.99| 2399.99|
|New DELL XPS 13 9...|   1499.99| 1799.99|
|Asus Zenbook Pro ...|   1799.99| 2159.99|
+--------------------+----------+--------+
only showing top 5 rows


**Explication :** Spark ne lance les transformations qu'au moment d'une action. Avant, il construit seulement un plan (visible avec `explain()`). Cela lui permet d'optimiser l'ensemble du traitement (par exemple, ne lire que les colonnes utiles) avant de l'exécuter.

## 4. Partitions et Spark UI

In [7]:
print("Nombre de partitions de commandes :", commandes.rdd.getNumPartitions())

# Répartition des lignes par partition
commandes.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()

# Exemple : redécouper en 4 partitions
commandes4 = commandes.repartition(4)
print("Après repartition(4) :", commandes4.rdd.getNumPartitions(), "partitions")
commandes4.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()

Nombre de partitions de commandes : 1


+---------+-----+
|partition|count|
+---------+-----+
|        0|  800|
+---------+-----+

Après repartition(4) : 4 partitions


+---------+-----+
|partition|count|
+---------+-----+
|        0|  200|
|        1|  200|
|        2|  200|
|        3|  200|
+---------+-----+



**Dans Spark UI (port 4040), à capturer pour le rendu :**
- **Jobs** : un job par action (show, count, collect…).
- **Stages** : un job est découpé en étapes ; une nouvelle étape commence quand les données doivent être redistribuées (shuffle, par exemple après `groupBy` ou `repartition`).
- **Tasks** : dans une étape, il y a une tâche par partition.
- **SQL / DataFrame** : le plan d'exécution de chaque requête.
- **Executors** : en mode local, un seul exécuteur (le driver) qui utilise les cœurs du Mac.
- **Environment** : la configuration (version de Spark, `local[*]`, connecteur MongoDB).

**Lien :** une **action** crée un **job** → le job est découpé en **stages** → chaque stage exécute une **task par partition**.

> Sur un petit jeu de données en mode local, les temps d'exécution ne sont pas représentatifs d'un cluster.

## Atelier 3 — Les indicateurs de BigData Commerce Analytics

### Indicateur 1 — Chiffre d'affaires par catégorie
Somme de `discounted_line_total` (montant des lignes après remise). On n'additionne pas `cart_total`, qui est répété sur chaque ligne du panier.

In [8]:
ca_categorie = (commandes
    .groupBy("category")
    .agg(F.round(F.sum("discounted_line_total"), 2).alias("chiffre_affaires"),
         F.count("*").alias("nb_lignes"))
    .orderBy(F.desc("chiffre_affaires")))
ca_categorie.show(10)

+------------------+----------------+---------+
|          category|chiffre_affaires|nb_lignes|
+------------------+----------------+---------+
|           vehicle|      1682064.05|       19|
|        motorcycle|       609460.61|       23|
|      mens-watches|       593063.86|       26|
|    womens-watches|       249189.51|       18|
|           laptops|        85006.24|       16|
|         furniture|        77674.97|       22|
|       smartphones|        73130.95|       60|
|mobile-accessories|        26160.99|       70|
|           tablets|        14890.16|       14|
|        mens-shoes|          8278.9|       23|
+------------------+----------------+---------+
only showing top 10 rows


**Commentaire :** le chiffre d'affaires total des lignes (après remise) est de 3 456 709,58 sur 24 catégories. Il est très concentré : `vehicle` représente à elle seule 48,7 %, et les 4 premières catégories (vehicle, motorcycle, mens-watches, womens-watches) 90,7 %. Ce sont des produits très chers achetés en petit nombre (19 lignes pour vehicle). À l'inverse, `kitchen-accessories` et `groceries` ont le plus de lignes mais un CA faible. *Ce CA compte toutes les lignes de panier, payées ou non.*

### Indicateur 2 — Top 10 clients
Somme des paiements **PAID** par `user_id` (on ajoute le `username` depuis les commandes pour lisibilité).

In [9]:
top10_clients = (payments
    .filter(F.col("status") == "PAID")
    .groupBy("user_id")
    .agg(F.round(F.sum("amount"), 2).alias("montant_encaisse"))
    .orderBy(F.desc("montant_encaisse"))
    .limit(10))
top10_clients.show()

+-------+----------------+
|user_id|montant_encaisse|
+-------+----------------+
|     30|       155507.16|
|     52|       130175.28|
|     83|        125679.0|
|     33|       125461.58|
|     45|       121385.95|
|    155|       110543.89|
|     78|       109890.08|
|    198|        92308.32|
|    194|        83615.86|
|    123|        78514.37|
+-------+----------------+



**Commentaire :** les 10 meilleurs clients totalisent 1 133 081,49, soit 48,8 % du montant encaissé (2 322 729,73) alors qu'ils ne représentent que 10 clients sur 208. Le premier,  (user_id 30), a payé 155 507,16. Comme chaque client n'a qu'un panier dans DummyJSON, ce classement reflète surtout la présence de produits chers (véhicules, montres) dans le panier.

### Indicateur 3 — Taux de refus
Nombre de paiements REFUSED / nombre total de tentatives.

In [10]:
repartition = (payments.groupBy("status")
    .agg(F.count("*").alias("nombre"), F.round(F.sum("amount"), 2).alias("montant"))
    .orderBy(F.desc("nombre")))
repartition.show()

total = payments.count()
refuses = payments.filter(F.col("status") == "REFUSED").count()
print(f"Taux de refus : {refuses} / {total} = {refuses / total:.1%}")

+-------+------+----------+
| status|nombre|   montant|
+-------+------+----------+
|   PAID|   145|2322729.73|
|REFUSED|    42| 915022.83|
|PENDING|    21| 218957.02|
+-------+------+----------+



Taux de refus : 42 / 208 = 20.2%


**Commentaire :** 42 paiements refusés sur 208 tentatives, soit un taux de refus de 20,2 %, très proche des 20 % fixés par la simulation. Les montants refusés (915 022,83) et en attente (218 957,02) ne sont pas encaissés : seul le montant PAID (2 322 729,73, 145 paiements) compte dans le CA encaissé.

### Indicateur 4 — Évolution mensuelle
Montants des paiements **PAID** regroupés par mois de `payment_date`.

> ⚠️ `payment_date` vient de `order_date`, une date **synthétique** générée par le script de collecte : cette évolution sert à pratiquer la méthode, pas à conclure sur une vraie saisonnalité.

In [11]:
evolution_mensuelle = (payments
    .filter(F.col("status") == "PAID")
    .withColumn("mois", F.date_format(F.to_date("payment_date"), "yyyy-MM"))
    .groupBy("mois")
    .agg(F.round(F.sum("amount"), 2).alias("montant_encaisse"),
         F.count("*").alias("nb_paiements"))
    .orderBy("mois"))
evolution_mensuelle.show(12)

+-------+----------------+------------+
|   mois|montant_encaisse|nb_paiements|
+-------+----------------+------------+
|2025-01|        94681.02|          11|
|2025-02|       242801.21|          10|
|2025-03|        306391.7|          19|
|2025-04|       273017.31|          10|
|2025-05|       207660.35|           7|
|2025-06|       251173.11|          13|
|2025-07|        56201.18|          12|
|2025-08|       147370.82|          13|
|2025-09|        94763.06|          12|
|2025-10|       208734.09|          14|
|2025-11|       236563.93|          11|
|2025-12|       203371.95|          13|
+-------+----------------+------------+



**Commentaire :** les 145 paiements PAID sont répartis sur les 12 mois de 2025, avec un maximum en mars (306 391,70) et un minimum en juillet (56 201,18). Ces écarts viennent du hasard de la date synthétique et de la présence ou non de produits très chers dans un mois donné : ils ne traduisent aucune vraie saisonnalité.

## Sauvegarde des indicateurs pour le jour 3
Les résultats sont enregistrés en CSV dans `data/indicateurs/` pour la visualisation du jour 3.

In [12]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "data" / "indicateurs"
OUT.mkdir(parents=True, exist_ok=True)

for nom, resultat in {"ca_par_categorie": ca_categorie, "top10_clients": top10_clients,
                      "repartition_paiements": repartition, "evolution_mensuelle": evolution_mensuelle}.items():
    resultat.toPandas().to_csv(OUT / f"{nom}.csv", index=False)
    print("Enregistré :", OUT / f"{nom}.csv")

Enregistré : /mnt/c/Users/mouad/Downloads/BigData_Commerce_Analytics_DummyJSON/BigData_Commerce_Analytics_DummyJSON/data/indicateurs/ca_par_categorie.csv
Enregistré : /mnt/c/Users/mouad/Downloads/BigData_Commerce_Analytics_DummyJSON/BigData_Commerce_Analytics_DummyJSON/data/indicateurs/top10_clients.csv


Enregistré : /mnt/c/Users/mouad/Downloads/BigData_Commerce_Analytics_DummyJSON/BigData_Commerce_Analytics_DummyJSON/data/indicateurs/repartition_paiements.csv


Enregistré : /mnt/c/Users/mouad/Downloads/BigData_Commerce_Analytics_DummyJSON/BigData_Commerce_Analytics_DummyJSON/data/indicateurs/evolution_mensuelle.csv


## Questions de synthèse

**Différence entre une collection MongoDB et un DataFrame Spark ?**
Une collection MongoDB **stocke** durablement des documents JSON, chacun pouvant avoir des champs différents. Un DataFrame Spark est une table **en mémoire, temporaire et distribuée**, avec un schéma fixe (colonnes typées), qui sert à **traiter** les données. Ici, MongoDB stocke et Spark calcule.

**Différence entre une transformation et une action ?**
Une transformation (`filter`, `groupBy`…) décrit une étape et ne calcule rien. Une action (`show`, `count`, `collect`…) demande un résultat et déclenche l'exécution.

**Pourquoi parle-t-on de lazy evaluation ?**
Parce que Spark est « paresseux » : il attend une action pour exécuter les transformations. Il peut ainsi optimiser tout le plan avant de le lancer.

**Rôle des partitions, Jobs, Stages et Tasks ?**
Les données sont découpées en **partitions** traitées en parallèle. Chaque action crée un **job**, découpé en **stages** (séparés par les redistributions de données), et chaque stage lance une **task par partition**.

**Pourquoi calculer le CA encaissé uniquement avec les paiements PAID ?**
Un paiement REFUSED n'a rapporté aucun argent, et un paiement PENDING n'est pas encore confirmé. Les compter gonflerait le chiffre d'affaires avec de l'argent que la boutique n'a pas reçu.

In [13]:
# À exécuter seulement à la fin, après les captures de Spark UI (Spark UI s'arrête avec la session)
# spark.stop()